In [14]:
import yfinance as yf
import pandas as pd
from pathlib import Path
import re

In [15]:

MX_DIR = Path("Mexico")
OUT_DIR = MX_DIR / "DatosHistoricos"
OUT_DIR.mkdir(parents=True, exist_ok=True)

xlsx_files = list(MX_DIR.glob("*.xlsx"))

# Agregar .MX automáticamente
tickers = [
    f"{file.stem.strip()}.MX"
    for file in xlsx_files
    if not file.name.startswith("~$")
]

print("Total tickers:", len(tickers))


Total tickers: 32


In [16]:

df = yf.download(
    tickers=tickers,
    period="20y",
    interval="1wk",
    auto_adjust=True,
    group_by="ticker",
    progress=False
)

df = df.reset_index()


if isinstance(df.columns, pd.MultiIndex):
    new_cols = []
    for col in df.columns:
        if col[0] == "Date":
            new_cols.append(("Fecha", "")) 
        else:
            new_cols.append(col)
    
    df.columns = pd.MultiIndex.from_tuples(new_cols)

df.columns = pd.MultiIndex.from_tuples(
    [(ticker, 
      "Apertura" if field.lower() == "open" else
      "Máximo" if field.lower() == "high" else
      "Mínimo" if field.lower() == "low" else
      "Cierre" if field.lower() == "close" else
      "Vol." if field.lower() == "volume" else field)
     for ticker, field in df.columns]
)

In [17]:
##exportar a csv

tickers_en_df = [t for t in df.columns.get_level_values(0).unique() if t != "Fecha"]

cols_req = ["Apertura", "Máximo", "Mínimo", "Cierre", "Vol."]

for t in tickers_en_df:

    cols = [("Fecha", "")] + [(t, c) for c in cols_req if (t, c) in df.columns]
    one = df.loc[:, cols].copy()

    one.columns = ["Fecha"] + [c for c in cols_req if (t, c) in df.columns]

    if "Cierre" in one.columns:
        one = one.dropna(subset=["Cierre"])

    out_path = OUT_DIR / f"Datos históricos {t}.csv"
    one.to_csv(out_path, index=False, encoding="utf-8-sig")

    print("Guardado:", out_path)

Guardado: Mexico\DatosHistoricos\Datos históricos KOFUBL.MX.csv
Guardado: Mexico\DatosHistoricos\Datos históricos GMXT.MX.csv
Guardado: Mexico\DatosHistoricos\Datos históricos VOLARA.MX.csv
Guardado: Mexico\DatosHistoricos\Datos históricos BBAJIOO.MX.csv
Guardado: Mexico\DatosHistoricos\Datos históricos GENTERA.MX.csv
Guardado: Mexico\DatosHistoricos\Datos históricos GFNORTEO.MX.csv
Guardado: Mexico\DatosHistoricos\Datos históricos PE&OLES.MX.csv
Guardado: Mexico\DatosHistoricos\Datos históricos RA.MX.csv
Guardado: Mexico\DatosHistoricos\Datos históricos KIMBERA.MX.csv
Guardado: Mexico\DatosHistoricos\Datos históricos GCARSOA1.MX.csv
Guardado: Mexico\DatosHistoricos\Datos históricos AC.MX.csv
Guardado: Mexico\DatosHistoricos\Datos históricos CHDRAUIB.MX.csv
Guardado: Mexico\DatosHistoricos\Datos históricos WALMEX.MX.csv
Guardado: Mexico\DatosHistoricos\Datos históricos ASURB.MX.csv
Guardado: Mexico\DatosHistoricos\Datos históricos LIVEPOLC-1.MX.csv
Guardado: Mexico\DatosHistoricos\Dato